# Model_VGG_Pretrained
VGG16 with Batch Normalization, fine-tuned from ImageNet weights on VWW using
three-phase progressive unfreezing.

**Three-phase protocol:**
- Phase 1 (epochs 1–9): backbone frozen, train head only at lr=3e-4
- Phase 2 (epochs 10–19): unfreeze features[24:] at lr=1e-4
- Phase 3 (epochs 20–30): unfreeze all features at lr=3e-5

**Note:** VGG16-BN has ~138M parameters — heavily over-parameterized for a
7,000-image binary task. Compare against `Model_VGG.ipynb` (scratch) to isolate
the effect of pretrained weights.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import os, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import VGG_Pretrained, count_params, model_size_mb
from utils.train   import setup_device, set_seed, evaluate, train_multi_seed, plot_history

device = setup_device(seed=41)

In [ ]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

In [ ]:
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints"

In [ ]:
# ── Parameter count before training ─────────────────────────────────
model_tmp = VGG_Pretrained().to(device)
total, trainable = count_params(model_tmp)
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}  (head only at start)")
print(f"Model size      : {model_size_mb(model_tmp):.1f} MB")
del model_tmp

In [ ]:
results, best = train_multi_seed(
    model_fn     = VGG_Pretrained,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    seeds        = [41, 52, 63, 74, 85],
    save_dir     = SAVE_DIR,
    name_prefix  = "vgg_pretrained",
    pretrained   = True,           # ← uses train_model_three_phase internally
    # Pretrained hyperparameters
    epochs          = 30,
    lr_phase1       = 3e-4,
    lr_phase2       = 1e-4,
    lr_phase3       = 3e-5,
    phase2_epoch    = 10,
    phase3_epoch    = 20,
    weight_decay    = 1e-4,
    label_smoothing = 0.1,
    patience        = 10,
)

In [ ]:
plot_history(best, title=f"VGG16-BN Pretrained (seed {best['seed']})")

accs = [r["best_acc"] for r in results]
print(f"\nVGG16-BN Pretrained  |  Mean: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%  |  Best: {best['best_acc']*100:.2f}% (seed {best['seed']})")
print(f"Best checkpoint: {best['save_path']}")

## Scratch vs Pretrained comparison
Run `Model_VGG.ipynb` first, then come back here to compare.

In [ ]:
# ── Compare scratch vs pretrained val accuracy ───────────────────────
# Load best scratch VGG checkpoint and evaluate on val set
SCRATCH_CKPT = f"{SAVE_DIR}/vggstyle_seed_85.pth"   # adjust seed

from utils.models import VWW_VGGStyle

if os.path.exists(SCRATCH_CKPT):
    scratch_model = VWW_VGGStyle().to(device)
    scratch_model.load_state_dict(torch.load(SCRATCH_CKPT, map_location=device))
    scratch_acc = evaluate(scratch_model, val_loader, device)

    pretrained_model = VGG_Pretrained().to(device)
    pretrained_model.load_state_dict(torch.load(best["save_path"], map_location=device))
    pretrained_acc = evaluate(pretrained_model, val_loader, device)

    print("="*50)
    print(f"VGG scratch    val acc: {scratch_acc*100:.2f}%")
    print(f"VGG pretrained val acc: {pretrained_acc*100:.2f}%")
    print(f"Pretrained gain       : {(pretrained_acc - scratch_acc)*100:+.2f}%")
    print("="*50)
else:
    print("⚠️  Run Model_VGG.ipynb first to get the scratch checkpoint")